# Initialization

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
PRODUCT_RENAME_MAP = {
    "prd_id": "product_id",
    "cat_id": "category_id",
    "prd_key": "product_key",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date"
}

# Reading from Bronze table

In [0]:
df = spark.table("workspace.bronze.crm_prd_info")

# Transformations

## Renaming the Columns

In [0]:
for old_name, new_name in PRODUCT_RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Extract Category ID

In [0]:
df = df.withColumn(
    "category_id",
    F.regexp_replace(
        F.substring("product_key", 1, 5),
        "-",
        "_"
    )
)

## Extract Product Key

In [0]:
df = df.withColumn(
    "product_key",
    F.substring(
        "product_key",
        7,
        100
    )
)

## Replace NULL Cost

In [0]:
df = df.withColumn(
    "product_cost",
    F.coalesce(
        F.col("product_cost"),
        F.lit(0)
    )
)

## Standardize Product Line

In [0]:
df = df.withColumn(
    "product_line",
    F.when(
        F.upper(F.trim(F.col("product_line"))) == "R",
        "Road"
    )
    .when(
        F.upper(F.trim(F.col("product_line"))) == "M",
        "Mountains"
    )
    .when(
        F.upper(F.trim(F.col("product_line"))) == "S",
        "Other Sales"
    )
    .when(
        F.upper(F.trim(F.col("product_line"))) == "T",
        "Touring"
    )
    .otherwise("Unknown")
)

## Convert to Date

In [0]:
df = df.withColumn(
    "start_date",
    F.to_date("start_date")
)

## Calculate End Date Using LEAD

In [0]:
window_spec = Window.partitionBy("product_key") \
                    .orderBy("start_date")

df = df.withColumn(
    "end_date",
    F.lead("start_date")
    .over(window_spec)
)

## Subtract One Day

In [0]:
df = df.withColumn(
    "end_date",
    F.date_sub(
        F.col("end_date"),
        1
    )
)

## Selecting the Columns

In [0]:
df = df.select(
    "product_id",
    "category_id",
    "product_key",
    "product_name",
    "product_cost",
    "product_line",
    "start_date",
    "end_date"
)

# Writing into Silver Table

In [0]:
(
    df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("silver.crm_products")
)